In [0]:
# ============================================================
# PASO 1: Verificación de training_images en Silver
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql import SparkSession

CATALOG       = "proyecto_smart_claims"
SILVER_SCHEMA = "silver"

spark = SparkSession.builder.getOrCreate()

# ------------------------------------------------------------
# Cargar la tabla
# ------------------------------------------------------------
training_images = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.training_images")

# ------------------------------------------------------------
# 1. Verificar que la tabla existe y tiene filas
# ------------------------------------------------------------
print("=" * 60)
print("1. EXISTENCIA Y TAMAÑO DE LA TABLA")
print("=" * 60)

total_filas = training_images.count()
print(f"Total de imágenes registradas: {total_filas:,}")

if total_filas == 0:
    raise ValueError("❌ La tabla training_images está vacía. Revisa la capa Silver antes de continuar.")
else:
    print("✅ La tabla tiene registros.")

# ------------------------------------------------------------
# 2. Verificar que las columnas esperadas existen
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("2. COLUMNAS DISPONIBLES EN LA TABLA")
print("=" * 60)

columnas_esperadas = ["path", "image_name", "label", "content"]
columnas_presentes = training_images.columns

print(f"Columnas en la tabla: {columnas_presentes}")

faltantes = [c for c in columnas_esperadas if c not in columnas_presentes]

if faltantes:
    print(f"\n⚠️  Columnas faltantes: {faltantes}")
    print("   Verifica que la capa Silver incluyó todas las columnas necesarias.")
else:
    print("\n✅ Todas las columnas esperadas están presentes.")

# ------------------------------------------------------------
# 3. Verificar que la columna 'content' existe y tiene datos
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("3. VALIDACIÓN DE COLUMNA BINARIA (content)")
print("=" * 60)

if "content" in columnas_presentes:
    nulos_content = training_images.filter(F.col("content").isNull()).count()
    print(f"Imágenes con content nulo:     {nulos_content:,}")
    print(f"Imágenes con content presente: {total_filas - nulos_content:,}")

    if nulos_content > 0:
        print("\n⚠️  Hay imágenes sin contenido binario. Muestra de paths afectados:")
        training_images.filter(F.col("content").isNull()) \
            .select("path", "image_name", "label") \
            .show(10, truncate=False)
    else:
        print("✅ Todas las imágenes tienen contenido binario.")
else:
    print("⚠️  La columna 'content' no existe en la tabla.")
    print("   Las imágenes fueron registradas solo por path, sin contenido binario cargado.")

# ------------------------------------------------------------
# 4. Verificar la columna 'label'
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("4. VALIDACIÓN DE LABELS")
print("=" * 60)

nulos_label  = training_images.filter(F.col("label").isNull() | (F.col("label") == "")).count()
print(f"Imágenes con label nulo o vacío: {nulos_label:,}")

if nulos_label > 0:
    print("\n❌ Hay imágenes sin label. Muestra de casos problemáticos:")
    training_images.filter(F.col("label").isNull() | (F.col("label") == "")) \
        .select("path", "image_name", "label") \
        .show(10, truncate=False)
    raise ValueError("❌ Existen labels vacíos. Corrige la capa Silver antes de continuar.")
else:
    print("✅ Todos los registros tienen label.")

# ------------------------------------------------------------
# 5. Distribución de labels
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("5. DISTRIBUCIÓN DE LABELS")
print("=" * 60)

distribucion = (
    training_images
    .groupBy("label")
    .agg(
        F.count("*").alias("cantidad"),
        F.round(F.count("*") / total_filas * 100, 2).alias("porcentaje_%")
    )
    .orderBy("label")
)

distribucion.show(truncate=False)

# Detectar desbalance severo (alguna clase con menos del 5%)
minimo_pct = distribucion.agg(F.min("porcentaje_%")).collect()[0][0]
if minimo_pct < 5.0:
    print(f"⚠️  Hay clases con menos del 5% de representación ({minimo_pct}%).")
    print("   Considera técnicas de balanceo antes de entrenar.")
else:
    print("✅ Distribución de clases balanceada.")

# ------------------------------------------------------------
# 6. Verificar nombres de archivo y paths
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("6. VALIDACIÓN DE PATHS E IMAGE_NAME")
print("=" * 60)

nulos_path       = training_images.filter(F.col("path").isNull()).count()
nulos_image_name = training_images.filter(
    F.col("image_name").isNull() | (F.col("image_name") == "")
).count()

print(f"Paths nulos:       {nulos_path:,}")
print(f"image_name nulos:  {nulos_image_name:,}")

if nulos_path == 0 and nulos_image_name == 0:
    print("✅ Todos los registros tienen path e image_name válidos.")
else:
    print("⚠️  Hay registros con path o image_name inválidos. Revisa la carga en Bronze.")

# Verificar extensiones esperadas
print("\n>>> Extensiones de archivo encontradas:")
training_images.withColumn(
    "extension",
    F.lower(F.regexp_extract(F.col("image_name"), r"\.(\w+)$", 1))
).groupBy("extension").count().orderBy("extension").show()

# ------------------------------------------------------------
# 7. Muestra final
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("7. MUESTRA DE REGISTROS")
print("=" * 60)

cols_muestra = [c for c in ["path", "image_name", "label"] if c in columnas_presentes]
training_images.select(*cols_muestra).show(10, truncate=False)

# ------------------------------------------------------------
# VEREDICTO FINAL
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("VEREDICTO FINAL")
print("=" * 60)

if total_filas > 0 and nulos_label == 0 and nulos_path == 0 and nulos_image_name == 0:
    print("✅ La tabla training_images está lista para entrenar.")
else:
    print("❌ La tabla NO está lista. Revisa los puntos marcados con ⚠️ o ❌ antes de continuar.")

In [0]:
# ============================================================
# PASO 2: Instalación e importación de librerías
# ============================================================

# ------------------------------------------------------------
# Instalación de librerías
# ------------------------------------------------------------
%pip install transformers datasets Pillow mlflow torch torchvision accelerate --quiet

# Reiniciar el kernel después de instalar para que los paquetes
# queden disponibles correctamente en el entorno de Databricks
dbutils.library.restartPython()

In [0]:
# ------------------------------------------------------------
# Imports — correr en celda separada después del restart
# ------------------------------------------------------------

# Procesamiento de imágenes
from PIL import Image
import io
import numpy as np

# Datasets y Transformers (HuggingFace)
from datasets import Dataset
from transformers import (
    AutoFeatureExtractor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer,
)

# PyTorch
import torch
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# MLflow
import mlflow
import mlflow.pytorch
from mlflow.tracking import MlflowClient

# PySpark
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

# Utilitarios
import os
import random

spark = SparkSession.builder.getOrCreate()

# ------------------------------------------------------------
# Verificación del entorno
# ------------------------------------------------------------
print("=" * 60)
print("VERIFICACIÓN DEL ENTORNO")
print("=" * 60)

# Versiones
import transformers
import datasets
import PIL

print(f"Python:          {__import__('sys').version.split()[0]}")
print(f"PyTorch:         {torch.__version__}")
print(f"Transformers:    {transformers.__version__}")
print(f"Datasets:        {datasets.__version__}")
print(f"Pillow:          {PIL.__version__}")
print(f"MLflow:          {mlflow.__version__}")
print(f"NumPy:           {np.__version__}")

# GPU disponible
print("\n" + "=" * 60)
print("DISPONIBILIDAD DE GPU")
print("=" * 60)

if torch.cuda.is_available():
    print(f"✅ GPU disponible: {torch.cuda.get_device_name(0)}")
    print(f"   Memoria total:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    DEVICE = "cuda"
else:
    print("⚠️  No hay GPU disponible. El entrenamiento usará CPU (más lento).")
    DEVICE = "cpu"

print(f"\nDispositivo activo: {DEVICE}")

# MLflow activo
print("\n" + "=" * 60)
print("MLFLOW")
print("=" * 60)

print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print("✅ MLflow listo.")

print("\n✅ Entorno listo para entrenar.")

In [0]:
# ============================================================
# PASO 3: Definir el experimento en MLflow
# ============================================================

import mlflow
from mlflow.tracking import MlflowClient

# ------------------------------------------------------------
# Nombre del experimento
# El experimento agrupa todas las corridas del entrenamiento.
# Si no existe, MLflow lo crea automáticamente.
# Si ya existe, MLflow lo reutiliza y agrega nuevas corridas.
# ------------------------------------------------------------

EXPERIMENT_NAME = "/Users/ancamihe@hotmail.com/databricks_repo/Proyecto_smart_claims/clasificacion_danios"

mlflow.set_experiment(EXPERIMENT_NAME)

# ------------------------------------------------------------
# Verificar que el experimento quedó registrado
# ------------------------------------------------------------
client    = MlflowClient()
experimento = client.get_experiment_by_name(EXPERIMENT_NAME)

print("=" * 60)
print("EXPERIMENTO MLFLOW")
print("=" * 60)
print(f"Nombre:      {experimento.name}")
print(f"ID:          {experimento.experiment_id}")
print(f"Estado:      {'✅ Activo' if experimento.lifecycle_stage == 'active' else '⚠️ ' + experimento.lifecycle_stage}")
print(f"Ubicación:   {experimento.artifact_location}")

print("""
============================================================
¿QUÉ ES UN EXPERIMENTO EN MLFLOW?
============================================================

  EXPERIMENTO: clasificacion_danios
  │
  ├── Corrida 1 → primer entrenamiento (ej: lr=2e-5, epochs=3)
  │     ├── Parámetros: learning_rate, num_epochs, batch_size...
  │     ├── Métricas:   accuracy, loss, eval_accuracy...
  │     ├── Artefactos: confusion matrix, gráficas...
  │     └── Modelo:     checkpoint guardado
  │
  ├── Corrida 2 → segundo entrenamiento (ej: lr=5e-5, epochs=5)
  │     └── ...
  │
  └── Corrida N → cada vez que se reentrene queda registrado aquí

  → Todas las corridas quedan trazables y comparables.
  → El mejor modelo se puede promover a producción desde MLflow.
============================================================
""")

print("✅ Experimento listo. Todas las corridas del entrenamiento")
print(f"   quedarán guardadas en: {EXPERIMENT_NAME}")

In [0]:
# ============================================================
# PASO 4: Preparar las imágenes para el entrenamiento
# ============================================================

from PIL import Image
import io
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.types import BinaryType
import mlflow

CATALOG       = "proyecto_smart_claims"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA   = "gold"

# Tamaño estándar que esperan los modelos de visión tipo ViT
IMAGE_SIZE = (224, 224)

# ------------------------------------------------------------
# Cargar la tabla Silver de imágenes de entrenamiento
# ------------------------------------------------------------
print("=" * 60)
print("CARGANDO IMÁGENES DESDE SILVER")
print("=" * 60)

training_images = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.training_images")

print(f"Total imágenes: {training_images.count():,}")
print(f"Columnas disponibles: {training_images.columns}")

# ------------------------------------------------------------
# Función de redimensionamiento
# Recibe bytes de imagen y devuelve bytes redimensionados
# No cambia el significado de la imagen, solo su tamaño
# ------------------------------------------------------------

def redimensionar_imagen(imagen_bytes: bytes, size: tuple = IMAGE_SIZE) -> bytes:
    """
    Abre la imagen desde bytes, la redimensiona al tamaño
    estándar requerido por el modelo y la devuelve como bytes.
    Convierte RGB si es necesario.
    """
    img = Image.open(io.BytesIO(imagen_bytes))
    
    # Convertir a RGB si la imagen está en otro modo (ej: RGBA, L)
    if img.mode != 'RGB':
        img = img.convert('RGB')
    
    # Redimensionar
    img_resized = img.resize(size, Image.Resampling.LANCZOS)
    
    # Convertir de vuelta a bytes
    buffer = io.BytesIO()
    img_resized.save(buffer, format='JPEG')
    return buffer.getvalue()

In [0]:
# ============================================================
# PASO 5: Construir el dataset de entrenamiento y validación
# ============================================================

from PIL import Image
from datasets import Dataset
import io
import numpy as np
import mlflow

CATALOG       = "proyecto_smart_claims"
SILVER_SCHEMA = "silver"

# Proporción de datos para validación (20%)
VAL_SIZE = 0.2
SEED     = 42

# ------------------------------------------------------------
# Cargar la tabla Silver con imágenes de entrenamiento
# ------------------------------------------------------------
print("=" * 60)
print("CARGANDO TABLA silver.training_images")
print("=" * 60)

training_ready = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.training_images")

total = training_ready.count()
print(f"Total imágenes disponibles: {total:,}")
training_ready.groupBy("label").count().orderBy("label").show()

# ------------------------------------------------------------
# Diccionario de clases
# Relaciona cada label (texto) con un identificador numérico
# El modelo no trabaja con texto sino con números
# ------------------------------------------------------------
print("=" * 60)
print("DICCIONARIO DE CLASES")
print("=" * 60)

labels_distintos = sorted([
    row["label"] for row in training_ready.select("label").distinct().collect()
])

# label → número (para entrenar)
label2id = {label: idx for idx, label in enumerate(labels_distintos)}

# número → label (para interpretar predicciones)
id2label = {idx: label for label, idx in label2id.items()}

print("Relación label → id numérico:")
for label, idx in label2id.items():
    print(f"  '{label}' → {idx}")

print(f"\nTotal de clases: {len(label2id)}")

# ------------------------------------------------------------
# Convertir la tabla Spark a lista de Python
# para construir el dataset de HuggingFace
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("CONVIRTIENDO TABLA A DATASET")
print("=" * 60)

filas = training_ready.select("image_name", "label", "content").collect()

def construir_registro(fila):
    """
    Convierte una fila de Spark en un diccionario con:
    - image:    objeto PIL listo para el modelo
    - label:    identificador numérico de la clase
    - label_str: nombre original del label (para referencia)
    """
    img = Image.open(io.BytesIO(fila["content"])).convert("RGB")
    return {
        "image":     img,
        "label":     label2id[fila["label"]],
        "label_str": fila["label"],
    }

registros = [construir_registro(f) for f in filas]
print(f"✅ {len(registros):,} registros construidos.")

# ------------------------------------------------------------
# Crear el dataset de HuggingFace y dividir
# ------------------------------------------------------------
print("\n" + "=" * 60)
print(f"DIVIDIENDO: {int((1-VAL_SIZE)*100)}% entrenamiento / {int(VAL_SIZE*100)}% validación")
print("=" * 60)

dataset_completo = Dataset.from_list(registros)

split = dataset_completo.train_test_split(
    test_size  = VAL_SIZE,
    seed       = SEED
)

dataset_train = split["train"]
dataset_val   = split["test"]

print(f"Imágenes de entrenamiento: {len(dataset_train):,}")
print(f"Imágenes de validación:    {len(dataset_val):,}")

# Verificar distribución en cada split
print("\nDistribución en entrenamiento:")
train_labels = dataset_train["label_str"]
for label in labels_distintos:
    n = train_labels.count(label)
    print(f"  {label}: {n:,} ({n/len(dataset_train)*100:.1f}%)")

print("\nDistribución en validación:")
val_labels = dataset_val["label_str"]
for label in labels_distintos:
    n = val_labels.count(label)
    print(f"  {label}: {n:,} ({n/len(dataset_val)*100:.1f}%)")

# ------------------------------------------------------------
# Registrar en MLflow
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("REGISTRANDO EN MLFLOW")
print("=" * 60)

with mlflow.start_run(run_name="preparacion_dataset", nested=True):
    mlflow.log_param("total_imagenes",       total)
    mlflow.log_param("val_size",             VAL_SIZE)
    mlflow.log_param("seed",                 SEED)
    mlflow.log_param("num_clases",           len(label2id))
    mlflow.log_param("clases",               str(labels_distintos))
    mlflow.log_metric("train_size",          len(dataset_train))
    mlflow.log_metric("val_size_real",       len(dataset_val))

print("✅ Parámetros del dataset registrados en MLflow.")

# ------------------------------------------------------------
# Resumen final
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("RESUMEN PASO 5")
print("=" * 60)
print(f"  Dataset entrenamiento:  {len(dataset_train):,} imágenes")
print(f"  Dataset validación:     {len(dataset_val):,} imágenes")
print(f"  Clases:                 {labels_distintos}")
print(f"  label2id:               {label2id}")
print(f"  id2label:               {id2label}")
print("\n✅ Paso 5 completo. dataset_train y dataset_val listos para el modelo.")

In [0]:
# ============================================================
# PASO 6: Convertir el binario a imagen
# ============================================================

from PIL import Image
from transformers import AutoImageProcessor
import io
import numpy as np

# ------------------------------------------------------------
# Modelo base que se usará para entrenar
# ViT (Vision Transformer) preentrenado en ImageNet
# El feature extractor sabe exactamente qué formato necesita
# ------------------------------------------------------------

MODEL_CHECKPOINT = "google/vit-base-patch16-224"

print("=" * 60)
print("CARGANDO IMAGE PROCESSOR")
print("=" * 60)

image_processor = AutoImageProcessor.from_pretrained(MODEL_CHECKPOINT)

print(f"Modelo base:       {MODEL_CHECKPOINT}")
print(f"Tamaño esperado:   {image_processor.size}")
print(f"Media (mean):      {image_processor.image_mean}")
print(f"Desviación (std):  {image_processor.image_std}")
print("✅ Image processor cargado.")

# ------------------------------------------------------------
# ¿Qué hace el image processor?
#
# El binario JPEG no puede entrar directamente al modelo.
# El image processor hace tres cosas:
#
#   1. Abre la imagen PIL y la convierte a array numérico
#   2. Normaliza los valores de píxel (0-255 → rango estándar)
#      usando la media y desviación del preentrenamiento
#   3. Devuelve tensores con la forma exacta que el modelo espera:
#      [canales, altura, anchura] → [3, 224, 224]
# ------------------------------------------------------------

# ------------------------------------------------------------
# Función de transformación
# Recibe un batch del dataset y devuelve los pixel_values
# que el modelo puede leer directamente
# ------------------------------------------------------------

def transformar_imagenes(batch):
    """
    Convierte el contenido binario de cada imagen en tensores
    normalizados listos para el modelo ViT.

    El dataset de HuggingFace llama esta función en batches,
    por eso recibe una lista de imágenes y no una sola.
    """
    imagenes_pil = []

    for item in batch["image"]:
        # Si ya es un objeto PIL (viene del paso anterior), usarlo directo
        if isinstance(item, Image.Image):
            img = item.convert("RGB")
        # Si todavía es binario, convertirlo a PIL primero
        elif isinstance(item, (bytes, bytearray)):
            img = Image.open(io.BytesIO(item)).convert("RGB")
        else:
            raise ValueError(f"Tipo de imagen no reconocido: {type(item)}")

        imagenes_pil.append(img)

    # El image processor normaliza y convierte a tensor
    encoded = image_processor(images=imagenes_pil, return_tensors="pt")

    # Agregar los pixel_values al batch
    batch["pixel_values"] = encoded["pixel_values"]

    return batch

# ------------------------------------------------------------
# Aplicar la transformación sobre train y validación
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("APLICANDO TRANSFORMACIÓN A LOS DATASETS")
print("=" * 60)

dataset_train_encoded = dataset_train.map(
    transformar_imagenes,
    batched      = True,
    batch_size   = 16,
    desc         = "Transformando entrenamiento",
)

dataset_val_encoded = dataset_val.map(
    transformar_imagenes,
    batched      = True,
    batch_size   = 16,
    desc         = "Transformando validación",
)

print(f"✅ Entrenamiento transformado: {len(dataset_train_encoded):,} imágenes")
print(f"✅ Validación transformada:    {len(dataset_val_encoded):,} imágenes")

# ------------------------------------------------------------
# Configurar formato para PyTorch
# Solo las columnas que el modelo necesita
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("CONFIGURANDO FORMATO PYTORCH")
print("=" * 60)

dataset_train_encoded.set_format(
    type    = "torch",
    columns = ["pixel_values", "label"]
)

dataset_val_encoded.set_format(
    type    = "torch",
    columns = ["pixel_values", "label"]
)

print("Columnas activas para el modelo: ['pixel_values', 'label']")
print("✅ Formato PyTorch configurado.")

# ------------------------------------------------------------
# Verificación visual: inspeccionar un ejemplo transformado
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("VERIFICACIÓN: INSPECCIONANDO UN EJEMPLO")
print("=" * 60)

ejemplo = dataset_train_encoded[0]

print(f"Claves disponibles:     {list(ejemplo.keys())}")
print(f"Forma de pixel_values:  {ejemplo['pixel_values'].shape}")
print(f"Tipo de tensor:         {ejemplo['pixel_values'].dtype}")
print(f"Label numérico:         {ejemplo['label'].item()} → '{id2label[ejemplo['label'].item()]}'")
print(f"Valor mínimo de píxel:  {ejemplo['pixel_values'].min():.4f}")
print(f"Valor máximo de píxel:  {ejemplo['pixel_values'].max():.4f}")

# Verificar que la normalización es correcta
# Los valores deben estar en rango aproximado [-3, 3] tras normalizar
if ejemplo["pixel_values"].min() >= -4 and ejemplo["pixel_values"].max() <= 4:
    print("\n✅ Normalización correcta. Valores en rango esperado.")
else:
    print("\n⚠️  Los valores de píxel están fuera del rango esperado.")
    print("   Verifica que el image processor corresponde al modelo.")

print("\n✅ Paso 6 completo.")
print("   dataset_train_encoded y dataset_val_encoded listos para el Trainer.")

In [0]:
# ============================================================
# PASO 7: Seleccionar y cargar el modelo base
# ============================================================

from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
)
import torch
import mlflow

# ------------------------------------------------------------
# ¿Por qué no entrenamos desde cero?
#
# Un modelo entrenado desde cero necesita millones de imágenes
# y semanas de cómputo para aprender a reconocer bordes,
# texturas, formas y objetos.
#
# ViT (Vision Transformer) ya fue entrenado con ImageNet
# (14 millones de imágenes, 1000 clases). Ya sabe reconocer
# patrones visuales generales.
#
# Lo que hacemos aquí se llama Fine-Tuning:
#   → Tomamos ese conocimiento general
#   → Reemplazamos solo la capa final de clasificación
#   → La adaptamos a nuestras clases de daños de vehículos
#   → Entrenamos pocas épocas para ajustar al nuevo problema
# ------------------------------------------------------------

MODEL_CHECKPOINT = "google/vit-base-patch16-224"

print("=" * 60)
print("MODELO BASE SELECCIONADO")
print("=" * 60)
print(f"Checkpoint:  {MODEL_CHECKPOINT}")
print(f"Arquitectura: Vision Transformer (ViT)")
print(f"Preentrenado en: ImageNet-21k")
print(f"Clases originales del modelo: 1000")
print(f"Clases de nuestro problema:   {len(label2id)}")

# ------------------------------------------------------------
# Cargar el Image Processor
# Es el procesador que prepara las imágenes para el modelo.
# Ya fue cargado en el Paso 6, pero se define aquí también
# para que este bloque sea independiente y claro.
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("CARGANDO IMAGE PROCESSOR")
print("=" * 60)

image_processor = AutoImageProcessor.from_pretrained(MODEL_CHECKPOINT)

print(f"✅ Image processor cargado.")
print(f"   Tamaño de imagen esperado: {image_processor.size}")
print(f"   Media de normalización:    {image_processor.image_mean}")
print(f"   Desviación estándar:       {image_processor.image_std}")

# ------------------------------------------------------------
# Cargar el modelo de clasificación
#
# ignore_mismatched_sizes=True le dice al modelo que está bien
# que la capa final tenga un tamaño distinto al original
# (1000 clases → nuestras clases), sin lanzar error.
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("CARGANDO MODELO DE CLASIFICACIÓN")
print("=" * 60)

model = AutoModelForImageClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels            = len(label2id),   # número de clases del problema
    id2label              = id2label,         # número → nombre de clase
    label2id              = label2id,         # nombre → número
    ignore_mismatched_sizes = True            # permite reemplazar la capa final
)

# Mover el modelo al dispositivo disponible (GPU o CPU)
model = model.to(DEVICE)

print(f"✅ Modelo cargado correctamente.")
print(f"   Dispositivo: {DEVICE}")
print(f"   Clases configuradas:")
for idx, label in id2label.items():
    print(f"     {idx} → '{label}'")

# ------------------------------------------------------------
# Inspeccionar la arquitectura del modelo
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("ARQUITECTURA DEL MODELO")
print("=" * 60)

# Parámetros totales
total_params = sum(p.numel() for p in model.parameters())

# Parámetros entrenables (los que se van a ajustar)
params_entrenables = sum(p.numel() for p in model.parameters() if p.requires_grad)

# Parámetros congelados (el conocimiento general que se conserva)
params_congelados = total_params - params_entrenables

print(f"Parámetros totales:      {total_params:,}")
print(f"Parámetros entrenables:  {params_entrenables:,}")
print(f"Parámetros congelados:   {params_congelados:,}")

# Mostrar solo la capa de clasificación (la que se reemplazó)
print(f"\nCapa de clasificación reemplazada:")
print(f"  {model.classifier}")

# ------------------------------------------------------------
# Verificar que el modelo hace una predicción de prueba
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("VERIFICACIÓN: PREDICCIÓN DE PRUEBA")
print("=" * 60)

model.eval()
with torch.no_grad():
    ejemplo        = dataset_train_encoded[0]
    pixel_values   = ejemplo["pixel_values"].unsqueeze(0).to(DEVICE)
    outputs        = model(pixel_values=pixel_values)
    logits         = outputs.logits
    pred_idx       = logits.argmax(dim=-1).item()
    pred_label     = id2label[pred_idx]
    label_real     = id2label[ejemplo["label"].item()]

print(f"Label real:      '{label_real}'")
print(f"Predicción:      '{pred_label}'  (antes de entrenar, puede ser incorrecta)")
print(f"Logits shape:    {logits.shape}  → un valor por cada clase")
print("\n⚠️  La predicción antes de entrenar es aleatoria. Esto es normal.")
print("   El Fine-Tuning del siguiente paso corregirá esto.")

# ------------------------------------------------------------
# Registrar en MLflow
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("REGISTRANDO EN MLFLOW")
print("=" * 60)

with mlflow.start_run(run_name="seleccion_modelo", nested=True):
    mlflow.log_param("model_checkpoint",     MODEL_CHECKPOINT)
    mlflow.log_param("arquitectura",         "ViT-base-patch16-224")
    mlflow.log_param("num_clases",           len(label2id))
    mlflow.log_param("dispositivo",          DEVICE)
    mlflow.log_param("total_params",         total_params)
    mlflow.log_param("params_entrenables",   params_entrenables)

print("✅ Parámetros del modelo registrados en MLflow.")

print("\n✅ Paso 7 completo.")
print(f"   Modelo '{MODEL_CHECKPOINT}' listo para Fine-Tuning")
print(f"   con {len(label2id)} clases: {list(label2id.keys())}")

In [0]:
# ============================================================
# PASO 8: Preprocesar las imágenes para el modelo
# ============================================================

from transformers import AutoImageProcessor
from PIL import Image
import torch
import numpy as np
import io

# Cargar el image processor
MODEL_CHECKPOINT = "google/vit-base-patch16-224"
image_processor = AutoImageProcessor.from_pretrained(MODEL_CHECKPOINT)

# ------------------------------------------------------------
# ¿Qué diferencia hay entre el Paso 6 y el Paso 8?
#
# Paso 6 → convirtió el binario a imagen PIL (abrir el archivo)
# Paso 8 → convierte la imagen PIL a tensores numéricos
#           normalizados con los valores exactos del modelo
#
# Es el último paso de preparación antes del entrenamiento.
# El Trainer llamará esta función automáticamente en cada batch.
# ------------------------------------------------------------

print("=" * 60)
print("FUNCIÓN DE PREPROCESAMIENTO")
print("=" * 60)

# ------------------------------------------------------------
# Función de preprocesamiento para el Trainer
#
# El Trainer de HuggingFace espera una función que reciba
# un batch del dataset y devuelva los campos procesados.
# Esta función se llama collate_fn o preprocessing_function.
# ------------------------------------------------------------

def preprocesar_batch(batch):
    """
    Recibe un batch con imágenes PIL y labels numéricos.
    Devuelve pixel_values normalizados listos para el modelo.

    El image_processor aplica:
      1. Redimensionamiento al tamaño esperado (224x224)
      2. Normalización por canal usando media y std de ImageNet
      3. Conversión a tensor PyTorch con forma [3, 224, 224]
    """
    imagenes = []

    for img in batch["image"]:
        # Aceptar PIL Image o binario por robustez
        if isinstance(img, Image.Image):
            imagenes.append(img.convert("RGB"))
        elif isinstance(img, (bytes, bytearray)):
            imagenes.append(
                Image.open(io.BytesIO(img)).convert("RGB")
            )
        else:
            raise ValueError(f"Formato no soportado: {type(img)}")

    # Image processor normaliza y convierte a tensor
    encoded = image_processor(images=imagenes, return_tensors="pt")
    batch["pixel_values"] = encoded["pixel_values"]

    return batch


# ------------------------------------------------------------
# Collate function
#
# El Trainer arma mini-batches durante el entrenamiento.
# Esta función le dice cómo agrupar múltiples ejemplos
# en un solo tensor listo para pasar al modelo.
# ------------------------------------------------------------

def collate_fn(ejemplos):
    """
    Agrupa una lista de ejemplos individuales en un batch.
    Apila los pixel_values y los labels en tensores únicos.
    """
    pixel_values = torch.stack([e["pixel_values"] for e in ejemplos])
    labels       = torch.tensor([e["label"] for e in ejemplos])

    return {
        "pixel_values": pixel_values,
        "labels":       labels,
    }

print("✅ preprocesar_batch definida.")
print("✅ collate_fn definida.")

# ------------------------------------------------------------
# Aplicar preprocesamiento sobre train y validación
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("APLICANDO PREPROCESAMIENTO")
print("=" * 60)

dataset_train_processed = dataset_train.map(
    preprocesar_batch,
    batched    = True,
    batch_size = 16,
    desc       = "Preprocesando entrenamiento",
)

dataset_val_processed = dataset_val.map(
    preprocesar_batch,
    batched    = True,
    batch_size = 16,
    desc       = "Preprocesando validación",
)

# Fijar formato PyTorch
dataset_train_processed.set_format(
    type    = "torch",
    columns = ["pixel_values", "label"]
)

dataset_val_processed.set_format(
    type    = "torch",
    columns = ["pixel_values", "label"]
)

print(f"✅ Entrenamiento preprocesado: {len(dataset_train_processed):,} imágenes")
print(f"✅ Validación preprocesada:    {len(dataset_val_processed):,} imágenes")

# ------------------------------------------------------------
# Verificación detallada de un ejemplo
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("VERIFICACIÓN DE UN EJEMPLO PREPROCESADO")
print("=" * 60)

ejemplo = dataset_train_processed[0]

print(f"Claves del ejemplo:      {list(ejemplo.keys())}")
print(f"Forma pixel_values:      {ejemplo['pixel_values'].shape}")
print(f"Tipo de dato:            {ejemplo['pixel_values'].dtype}")
print(f"Label:                   {ejemplo['label'].item()} → '{id2label[ejemplo['label'].item()]}'")
print(f"Valor mínimo de píxel:   {ejemplo['pixel_values'].min():.4f}")
print(f"Valor máximo de píxel:   {ejemplo['pixel_values'].max():.4f}")
print(f"Media de píxel:          {ejemplo['pixel_values'].mean():.4f}")

# ------------------------------------------------------------
# Verificación del collate_fn con un mini-batch de prueba
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("VERIFICACIÓN DEL COLLATE_FN")
print("=" * 60)

mini_batch  = [dataset_train_processed[i] for i in range(4)]
batch_listo = collate_fn(mini_batch)

print(f"pixel_values shape: {batch_listo['pixel_values'].shape}")
print(f"  → [batch_size=4, canales=3, alto=224, ancho=224]")
print(f"labels shape:       {batch_listo['labels'].shape}")
print(f"labels values:      {batch_listo['labels'].tolist()}")
print(f"  → {[id2label[l] for l in batch_listo['labels'].tolist()]}")
print(f"\n✅ El collate_fn arma batches correctamente.")

# ------------------------------------------------------------
# Resumen de lo que entra y sale
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("RESUMEN: QUÉ ENTRA Y QUÉ SALE")
print("=" * 60)
print("""
  ENTRADA (imagen PIL)
    → Objeto Image RGB de 224x224 px
    → Valores de píxel entre 0 y 255

  PROCESAMIENTO (image_processor)
    → Normalización canal a canal con media y std de ImageNet
    → Conversión a tensor float32

  SALIDA (pixel_values)
    → Tensor de forma [3, 224, 224]
    → Valores en rango aproximado [-3, 3]
    → Listo para entrar directamente al modelo ViT
""")

print("✅ Paso 8 completo.")
print("   dataset_train_processed, dataset_val_processed")
print("   y collate_fn listos para el Trainer del Paso 9.")

In [0]:
# ============================================================
# PASO 9: Configurar los parámetros de entrenamiento
# ============================================================

from transformers import TrainingArguments
import numpy as np
import mlflow
import torch


In [0]:

# ------------------------------------------------------------
# ¿Qué es cada parámetro?
#
# ÉPOCA (epoch): una pasada completa por todos los datos
#   → con pocas imágenes, 5-10 épocas suele ser suficiente
#
# BATCH SIZE: cuántas imágenes se procesan a la vez
#   → más grande = más memoria GPU, más estable el gradiente
#   → más pequeño = menos memoria, más ruido en el aprendizaje
#
# LEARNING RATE: qué tan grandes son los ajustes del modelo
#   → muy alto = el modelo no converge, salta sin aprender
#   → muy bajo = el modelo aprende muy lento
#   → 2e-5 es un valor estándar para Fine-Tuning con ViT
#
# WARMUP: épocas iniciales con lr muy bajo antes de subir
#   → evita que el modelo rompa los pesos preentrenados
#   → al principio el modelo es sensible, necesita ajustes suaves
# ------------------------------------------------------------

import os

# Limpiar variables de entorno de distributed training ANTES de crear TrainingArguments
for var in ["RANK", "WORLD_SIZE", "MASTER_ADDR", "MASTER_PORT", "LOCAL_RANK"]:
    os.environ.pop(var, None)

OUTPUT_DIR = "/tmp/smart_claims_vit"

print("=" * 60)
print("CONFIGURACIÓN DE ENTRENAMIENTO - MODO AGRESIVO")
print("=" * 60)
print("Se aplicarán parámetros optimizados para dataset pequeño:")
print("  • Épocas: 20 (más iteraciones de aprendizaje)")
print("  • Batch size: 4 (4x más actualizaciones de gradiente)")
print("  • Warmup steps: 10 (inicio gradual del learning rate)")
print("  • Learning rate: 5e-5 (ligeramente más alto)")
print("\n⏱️  Tiempo estimado: 20-30 minutos")
print("=" * 60)

training_args = TrainingArguments(
    output_dir = OUTPUT_DIR,

    # ── Épocas: aumentado a 20 para más aprendizaje ──────────
    num_train_epochs = 20,

    # ── Batch size: reducido a 4 para más actualizaciones ────
    per_device_train_batch_size = 4,
    per_device_eval_batch_size  = 4,

    # ── Learning rate: ligeramente más alto ──────────────────
    learning_rate          = 5e-5,
    lr_scheduler_type      = "cosine",
    warmup_steps           = 10,  # 10 pasos de warmup gradual

    eval_strategy          = "epoch",
    save_strategy          = "epoch",
    load_best_model_at_end = True,
    metric_for_best_model  = "accuracy",
    greater_is_better      = True,

    logging_strategy       = "epoch",
    report_to              = "none",

    seed                   = 42,

    fp16                   = torch.cuda.is_available(),
    dataloader_num_workers = 0,
    remove_unused_columns  = False,
)

# Configura logging_dir por variable de entorno
os.environ["TENSORBOARD_LOGGING_DIR"] = f"{OUTPUT_DIR}/logs"

print("\n✅ TrainingArguments configurado con parámetros agresivos.")

In [0]:


report_to = "mlflow", # enviar métricas directamente a MLflow
# ------------------------------------------------------------
# Mostrar la configuración completa
# ------------------------------------------------------------
print("=" * 60)
print("CONFIGURACIÓN DE ENTRENAMIENTO")
print("=" * 60)
print(f"  Épocas:                   {training_args.num_train_epochs}")
print(f"  Batch train:              {training_args.per_device_train_batch_size}")
print(f"  Batch eval:               {training_args.per_device_eval_batch_size}")
print(f"  Learning rate:            {training_args.learning_rate}")
print(f"  LR scheduler:             {training_args.lr_scheduler_type}")
print(f"  Warmup ratio:             {training_args.warmup_ratio}")
print(f"  Evaluación:               por {training_args.eval_strategy}")
print(f"  Mejor modelo según:       {training_args.metric_for_best_model}")
print(f"  Precisión mixta (fp16):   {training_args.fp16}")
print(f"  Directorio de salida:     {training_args.output_dir}")
print(f"  Reporte a:                MLflow")

# ------------------------------------------------------------
# Función de métricas
#
# El Trainer llama esta función al final de cada época
# con las predicciones y los labels reales.
# Debe devolver un diccionario con las métricas a registrar.
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("FUNCIÓN DE MÉTRICAS")
print("=" * 60)

def compute_metrics(eval_pred):
    """
    Calcula accuracy a partir de las predicciones del modelo.

    eval_pred contiene:
      - logits: puntuación cruda por clase  [n_ejemplos, n_clases]
      - labels: clase real de cada ejemplo  [n_ejemplos]
    """
    logits, labels = eval_pred

    # La clase predicha es la de mayor puntuación
    predicciones = np.argmax(logits, axis=-1)

    # Accuracy: porcentaje de predicciones correctas
    accuracy = (predicciones == labels).mean()

    # Accuracy por clase
    accuracy_por_clase = {}
    for idx, nombre in id2label.items():
        mask = labels == idx
        if mask.sum() > 0:
            acc_clase = (predicciones[mask] == labels[mask]).mean()
            accuracy_por_clase[f"accuracy_{nombre}"] = round(float(acc_clase), 4)

    return {
        "accuracy": round(float(accuracy), 4),
        **accuracy_por_clase,
    }

print("✅ compute_metrics definida.")
print("   Métricas que calcula:")
print("   → accuracy global")
print("   → accuracy por cada clase de daño")


In [0]:

# ------------------------------------------------------------
# Estimación de pasos de entrenamiento
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("ESTIMACIÓN DE PASOS")
print("=" * 60)

pasos_por_epoca = len(dataset_train_processed) // training_args.per_device_train_batch_size
pasos_totales   = pasos_por_epoca * training_args.num_train_epochs
pasos_warmup    = int(pasos_totales * (training_args.warmup_ratio or 0))

print(f"  Imágenes de entrenamiento: {len(dataset_train_processed):,}")
print(f"  Pasos por época:           {pasos_por_epoca:,}")
print(f"  Pasos totales:             {pasos_totales:,}")
print(f"  Pasos de warmup:           {pasos_warmup:,}")

# ------------------------------------------------------------
# Registrar configuración en MLflow
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("REGISTRANDO EN MLFLOW")
print("=" * 60)

with mlflow.start_run(run_name="configuracion_entrenamiento", nested=True):
    mlflow.log_param("num_train_epochs",            training_args.num_train_epochs)
    mlflow.log_param("per_device_train_batch_size", training_args.per_device_train_batch_size)
    mlflow.log_param("per_device_eval_batch_size",  training_args.per_device_eval_batch_size)
    mlflow.log_param("learning_rate",               training_args.learning_rate)
    mlflow.log_param("lr_scheduler_type",           training_args.lr_scheduler_type)
    mlflow.log_param("warmup_ratio",                training_args.warmup_ratio)
    mlflow.log_param("seed",                        training_args.seed)
    mlflow.log_param("fp16",                        training_args.fp16)
    mlflow.log_param("pasos_totales",               pasos_totales)
    mlflow.log_param("pasos_warmup",                pasos_warmup)

print("✅ Configuración registrada en MLflow.")

print("\n✅ Paso 9 completo.")
print("   training_args y compute_metrics listos para el Trainer.")

In [0]:
# ============================================================
# PASO 10: Entrenar el modelo con Fine-Tuning
# ============================================================

from transformers import Trainer
import mlflow

print("=" * 60)
print("INICIALIZANDO TRAINER")
print("=" * 60)

# ------------------------------------------------------------
# Crear el Trainer
#
# El Trainer orquesta todo el proceso de entrenamiento:
#   → Carga batches usando collate_fn
#   → Pasa los datos al modelo
#   → Calcula la pérdida
#   → Actualiza los pesos mediante backpropagation
#   → Evalúa en cada época usando compute_metrics
#   → Guarda checkpoints
#   → Registra métricas en MLflow
# ------------------------------------------------------------

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = dataset_train_processed,
    eval_dataset    = dataset_val_processed,
    data_collator   = collate_fn,
    compute_metrics = compute_metrics,
)

print("✅ Trainer inicializado correctamente.")
print(f"   Modelo:           {MODEL_CHECKPOINT}")
print(f"   Datos train:      {len(dataset_train_processed)} imágenes")
print(f"   Datos validación: {len(dataset_val_processed)} imágenes")
print(f"   Épocas:           {training_args.num_train_epochs}")
print(f"   Batch size:       {training_args.per_device_train_batch_size}")

# ------------------------------------------------------------
# Iniciar entrenamiento dentro de un run de MLflow
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("INICIANDO ENTRENAMIENTO")
print("=" * 60)
print("⏳ Este proceso puede tardar varios minutos...")
print()

with mlflow.start_run(run_name="entrenamiento_vit", nested=True):
    
    # Registrar parámetros clave
    mlflow.log_param("model", MODEL_CHECKPOINT)
    mlflow.log_param("train_size", len(dataset_train_processed))
    mlflow.log_param("val_size", len(dataset_val_processed))
    mlflow.log_param("epochs", training_args.num_train_epochs)
    mlflow.log_param("batch_size", training_args.per_device_train_batch_size)
    mlflow.log_param("learning_rate", training_args.learning_rate)
    
    # Entrenar
    train_result = trainer.train()
    
    # Registrar métricas finales de entrenamiento
    mlflow.log_metrics({
        "train_loss_final": train_result.training_loss,
        "train_steps": train_result.global_step,
    })
    
    print("\n" + "=" * 60)
    print("ENTRENAMIENTO COMPLETADO")
    print("=" * 60)
    print(f"   Pérdida final:    {train_result.training_loss:.4f}")
    print(f"   Pasos totales:    {train_result.global_step}")
    print(f"   Tiempo total:     {train_result.metrics.get('train_runtime', 'N/A')} seg")

# ------------------------------------------------------------
# Evaluación final en el conjunto de validación
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("EVALUACIÓN FINAL")
print("=" * 60)

eval_results = trainer.evaluate()

print(f"   Pérdida validación: {eval_results.get('eval_loss', 'N/A'):.4f}")
print(f"   Accuracy:           {eval_results.get('eval_accuracy', 'N/A'):.4f}")
print()
print("   Accuracy por clase:")
for key, value in eval_results.items():
    if key.startswith('eval_accuracy_'):
        clase = key.replace('eval_accuracy_', '')
        print(f"     • {clase}: {value:.4f}")

# Registrar resultados de evaluación en MLflow
with mlflow.start_run(run_name="evaluacion_final", nested=True):
    for key, value in eval_results.items():
        if isinstance(value, (int, float)):
            mlflow.log_metric(key, value)

print("\n✅ Evaluación completada y registrada en MLflow.")

# ------------------------------------------------------------
# Guardar el modelo entrenado
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("GUARDANDO MODELO")
print("=" * 60)

model_save_path = f"{OUTPUT_DIR}/modelo_final"
trainer.save_model(model_save_path)

print(f"✅ Modelo guardado en: {model_save_path}")
print(f"   El modelo incluye:")
print(f"     • Pesos del modelo entrenado")
print(f"     • Configuración del modelo")
print(f"     • Mapeos id2label y label2id")

print("\n" + "=" * 60)
print("✅ PASO 10 COMPLETO")
print("=" * 60)
print(f"Modelo '{MODEL_CHECKPOINT}' fine-tuneado exitosamente.")
print(f"Accuracy final: {eval_results.get('eval_accuracy', 'N/A'):.2%}")
print(f"\nPróximos pasos:")
print(f"  1. Revisar métricas en MLflow")
print(f"  2. Probar predicciones con nuevas imágenes")
print(f"  3. Registrar modelo en Unity Catalog si el desempeño es bueno")

In [0]:
# ============================================================
# PASO 11: Construir el pipeline de inferencia
# ============================================================

from transformers import pipeline, AutoImageProcessor, AutoModelForImageClassification
from PIL import Image
import io
import torch
import mlflow
import mlflow.pytorch
from mlflow.tracking import MlflowClient
from pyspark.sql import functions as F

CATALOG       = "proyecto_smart_claims"
SILVER_SCHEMA = "silver"
MODEL_CHECKPOINT = "google/vit-base-patch16-224"
OUTPUT_DIR       = "/tmp/smart_claims_vit"
EXPERIMENT_NAME = "/Users/ancamihe@hotmail.com/databricks_repo/Proyecto_smart_claims/clasificacion_danios"

# ------------------------------------------------------------
# Reconstruir diccionario de clases
# (por si el clúster se reinició y las variables se perdieron)
# ------------------------------------------------------------
print("=" * 60)
print("RECONSTRUYENDO DICCIONARIO DE CLASES")
print("=" * 60)

labels_distintos = sorted([
    row["label"] for row in
    spark.table(f"{CATALOG}.{SILVER_SCHEMA}.training_images")
    .select("label").distinct().collect()
])

label2id = {label: idx for idx, label in enumerate(labels_distintos)}
id2label  = {idx: label for label, idx in label2id.items()}

print(f"Clases encontradas: {labels_distintos}")
print(f"Total clases:       {len(label2id)}")

# ------------------------------------------------------------
# Cargar el modelo — intenta desde /tmp/ primero,
# si no existe lo carga desde MLflow
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("CARGANDO MODELO ENTRENADO")
print("=" * 60)

import os

if os.path.exists(f"{OUTPUT_DIR}/modelo_final"):
    print(f"✅ Modelo encontrado en {OUTPUT_DIR}/modelo_final")
    print("   Cargando desde disco local...")
    model = AutoModelForImageClassification.from_pretrained(
        f"{OUTPUT_DIR}/modelo_final",
        local_files_only   = True,
        num_labels         = len(label2id),
        id2label           = id2label,
        label2id           = label2id,
        ignore_mismatched_sizes = True,
    )
else:
    print(f"⚠️  Modelo no encontrado en {OUTPUT_DIR}/modelo_final")
    print("   Cargando desde MLflow...")
    try:
        model = mlflow.pytorch.load_model(
            "models:/smart_claims_clasificador_danios/latest"
        )
        print("✅ Modelo cargado desde MLflow.")
    except Exception as e:
        print(f"❌ No se pudo cargar desde MLflow: {e}")
        print("   Cargando modelo base sin fine-tuning como fallback...")
        model = AutoModelForImageClassification.from_pretrained(
            MODEL_CHECKPOINT,
            num_labels              = len(label2id),
            id2label                = id2label,
            label2id                = label2id,
            ignore_mismatched_sizes = True,
        )
        print("⚠️  Se cargó el modelo base SIN fine-tuning.")
        print("   Las predicciones no serán confiables.")

# Determinar dispositivo
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model  = model.to(DEVICE)
model.eval()

print(f"✅ Modelo listo en dispositivo: {DEVICE}")

# ------------------------------------------------------------
# Cargar el image processor
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("CARGANDO IMAGE PROCESSOR")
print("=" * 60)

image_processor = AutoImageProcessor.from_pretrained(MODEL_CHECKPOINT)
print(f"✅ Image processor cargado.")
print(f"   Tamaño esperado: {image_processor.size}")

# ------------------------------------------------------------
# Crear el pipeline de clasificación
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("CONSTRUYENDO PIPELINE DE INFERENCIA")
print("=" * 60)

clasificador = pipeline(
    task             = "image-classification",
    model            = model,
    image_processor  = image_processor,
    device           = 0 if torch.cuda.is_available() else -1,
)

print("✅ Pipeline de inferencia creado.")
print(f"   Tarea:       image-classification")
print(f"   Modelo:      {MODEL_CHECKPOINT}")
print(f"   Dispositivo: {'GPU' if torch.cuda.is_available() else 'CPU'}")
print(f"   Clases:      {list(label2id.keys())}")

# ------------------------------------------------------------
# Funciones auxiliares
# ------------------------------------------------------------

def predecir_imagen(imagen_input, top_k: int = None):
    """
    Recibe una imagen en bytes, PIL o ruta
    y devuelve las predicciones ordenadas por confianza.
    """
    if top_k is None:
        top_k = len(label2id)

    if isinstance(imagen_input, (bytes, bytearray)):
        img = Image.open(io.BytesIO(imagen_input)).convert("RGB")
    elif isinstance(imagen_input, Image.Image):
        img = imagen_input.convert("RGB")
    elif isinstance(imagen_input, str):
        img = Image.open(imagen_input).convert("RGB")
    else:
        raise ValueError(f"Formato no soportado: {type(imagen_input)}")

    return clasificador(img, top_k=top_k)


def mostrar_prediccion(resultados: list, imagen_nombre: str = "", label_real: str = ""):
    """
    Imprime los resultados de predicción de forma legible
    con barra visual de confianza.
    """
    titulo = f"PREDICCIÓN: {imagen_nombre}" if imagen_nombre else "PREDICCIÓN"
    print(f"\n{'─' * 55}")
    print(titulo)
    if label_real:
        print(f"Label real: '{label_real}'")
    print(f"{'─' * 55}")
    for i, r in enumerate(resultados):
        barra   = "█" * int(r["score"] * 30)
        espacio = "░" * (30 - len(barra))
        es_top  = " ← TOP 1" if i == 0 else ""
        print(f"  {r['label']:20s} {r['score']:6.2%}  {barra}{espacio}{es_top}")
    print(f"{'─' * 55}")
    if resultados:
        top1        = resultados[0]["label"]
        es_correcto = top1 == label_real if label_real else None
        if es_correcto is not None:
            print(f"  ¿Top-1 correcto?: {'✅ Sí' if es_correcto else '❌ No'}")


print("\n✅ Funciones predecir_imagen() y mostrar_prediccion() definidas.")

# ------------------------------------------------------------
# Prueba del pipeline con imágenes reales de Silver
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("PRUEBA DEL PIPELINE CON IMÁGENES DE VALIDACIÓN")
print("=" * 60)

muestra_val = (
    spark.table(f"{CATALOG}.{SILVER_SCHEMA}.training_images")
    .filter(F.col("content").isNotNull())
    .limit(3)
    .collect()
)

if not muestra_val:
    print("⚠️  No se encontraron imágenes con contenido binario en Silver.")
else:
    correctas = 0
    for fila in muestra_val:
        try:
            resultados = predecir_imagen(fila["content"])
            mostrar_prediccion(
                resultados,
                imagen_nombre = fila["image_name"],
                label_real    = fila["label"]
            )
            if resultados[0]["label"] == fila["label"]:
                correctas += 1
        except Exception as e:
            print(f"❌ Error procesando {fila['image_name']}: {e}")

    print(f"\n  Accuracy en muestra: {correctas}/{len(muestra_val)} "
          f"({correctas/len(muestra_val):.0%})")

# ------------------------------------------------------------
# Registrar en MLflow
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("REGISTRANDO PIPELINE EN MLFLOW")
print("=" * 60)

mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name="pipeline_inferencia"):
    mlflow.log_param("pipeline_task",  "image-classification")
    mlflow.log_param("modelo_base",    MODEL_CHECKPOINT)
    mlflow.log_param("num_clases",     len(label2id))
    mlflow.log_param("clases",         str(list(label2id.keys())))
    mlflow.log_param("dispositivo",    DEVICE)

    try:
        mlflow.pytorch.log_model(
            pytorch_model         = model,
            artifact_path         = "modelo_clasificacion_danios",
            registered_model_name = "smart_claims_clasificador_danios",
        )
        print("✅ Modelo registrado en MLflow como:")
        print("   'smart_claims_clasificador_danios'")
    except Exception as e:
        print(f"⚠️  No se pudo registrar el modelo en MLflow: {e}")

# ------------------------------------------------------------
# Resumen
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("RESUMEN DEL PIPELINE DE INFERENCIA")
print("=" * 60)
print("""
  ENTRADA
    → Imagen en bytes, PIL o ruta de archivo

  PREPROCESAMIENTO (automático)
    → Conversión a RGB
    → Redimensionamiento a 224x224
    → Normalización con media y std de ImageNet

  MODELO
    → Forward pass por el ViT fine-tuneado
    → Logits por cada clase de daño

  SALIDA
    → Lista de clases con score de confianza
    → Ordenada de mayor a menor probabilidad
""")

print("✅ Paso 11 completo.")
print(f"   Pipeline 'clasificador' listo para predecir.")
print(f"   Clases disponibles: {list(label2id.keys())}")